In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 23:36:48.452305: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 23:36:49.163975: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 23:36:50,215 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 23:36:50,217 [DEBUG] [Rain] Rain is initialized
2023-07-05 23:36:50,218 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 23:36:50,219 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 23:36:50,220 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 23:36:50,221 [DEBUG] [LazyProvisioner] Provisioner is initialized
2023-07-05 23:36:50,223 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 23:36:50,224 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 23:36:50,226 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 23:36:50,237 [INFO] [Provisioner] provisioner is serving
2023-07-05 23:36:50,238 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 23:36:50,240 [INFO] [Coordinator] coordinator is serving
2023-07-05 23:36:50,241 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 23:36:50,245 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 23:36:50,247 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 23:36:50,248 [DEBUG] [LazyProvisioner] Creating 3 workers
2023-07-05 23:36:50,249 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
2023-07-05 23:36:50,250 [DEBUG] [DividerAmbassador] divider ambassador is serving
2023-07-05 23:36:50,251 [DEBUG] [DividerProxy] Training Started
2023-07-05 23:36:50,436 [DEBUG] [Coordinator] coordinator is sending worker

Error in receiving the gradients from the workers: [Errno 2] No such file or directory: '../../..//RainData/divider/3_3_trained.pkl'


2023-07-05 23:36:51,255 [DEBUG] [DeepLearning] Error in calculating the new weights: 'NoneType' object is not subscriptable
2023-07-05 23:36:51,320 [DEBUG] [DeepLearning] Iteration 2/3 complete for worker 3.
2023-07-05 23:36:51,322 [DEBUG] [DeepLearning] Starting iteration 3/3
2023-07-05 23:36:51,323 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-05 23:36:51,325 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration3 to worker3
2023-07-05 23:36:51,325 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/3.pkl to worker3
2023-07-05 23:36:51,327 [ERROR] [DividerAmbassador] Error sending the model to the worker: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50153: Failed to connect to remote host: Connection refused"
	debug_error_string = "UNKNOWN:failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50153: Failed to co

Error in receiving the gradients from the workers: [Errno 2] No such file or directory: '../../..//RainData/divider/3_3_trained.pkl'
Error in receiving the gradients from the workers: [Errno 2] No such file or directory: '../../..//RainData/divider/3_3_trained.pkl'


2023-07-05 23:36:51,425 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-05 23:36:51,426 [DEBUG] [DividerAmbassador] divider begins executing iteration1 for worker2
2023-07-05 23:36:59,470 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-05 23:36:59,471 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-05 23:36:59,484 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-05 23:36:59,503 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
2023-07-05 23:36:59,531 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 2.
2023-07-05 23:36:59,533 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-05 23:36:59,533 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-05 23:36:59,534 [DEBUG] [DividerAmbassador] divider begins will 

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0859 - accuracy: 0.9738

Test accuracy: 97.4%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 23:37:13,427 [INFO] [Provisioner] provisioner is serving
2023-07-05 23:37:13,428 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 23:37:13,430 [INFO] [Coordinator] coordinator is serving
2023-07-05 23:37:13,431 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 23:37:13,433 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 23:37:13,434 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 23:37:13,435 [DEBUG] [LazyProvisioner] Creating 3 workers
2023-07-05 23:37:13,436 [DEBUG] [Provisioner] [Created workers]
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
2023-07-05 23:37:13,437 [DEBUG] [DividerAmbassador] divider ambassador is serving
2023-07-05 23:37:13,438 [DEBUG] [DividerProxy] Training Started
2023-07-05 23:37:13,503 [DEBUG] [Coordinator] coordinator is sending worker

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0859 - accuracy: 0.9738

Test accuracy: 97.4%
